In [14]:
import requests
import json
import os
from dotenv import load_dotenv

In [15]:
# Carga las variables del archivo .env al entorno del sistema
load_dotenv()

# --- EL API se encuentra en el archivo oculto
API_TOKEN = os.getenv("AQI_API_KEY")
BASE_URL = "https://api.waqi.info"

### 1.- Consultar calidad del aire por ciudad

In [16]:
# Nombre de la ciudad que queremos consultar
ciudad = "mexico-city"

# Construimos la URL completa para la petición
url_ciudad = f"{BASE_URL}/feed/{ciudad}/?token={API_TOKEN}"

# Hacemos la petición GET a la API
response = requests.get(url_ciudad)

# Convertimos la respuesta a formato JSON para poder manejarla fácilmente
data_ciudad = response.json()

# Imprimimos los datos de forma legible
import json
print(json.dumps(data_ciudad, indent=2))

{
  "status": "ok",
  "data": {
    "aqi": 47,
    "idx": 404,
    "attributions": [
      {
        "url": "http://www.aire.df.gob.mx/",
        "name": "SINAICA - Sistema Nacional de Informaci\u00f3n de la Calidad del Aire en Mexico",
        "logo": "Mexico-CuidaMexico.png"
      },
      {
        "url": "http://sinaica.inecc.gob.mx/",
        "name": "INECC - Instituto Nacional de Ecolog&iacute;a y Cambio Clim&aacute;tico",
        "logo": "Mexico-INECC.png"
      },
      {
        "url": "https://waqi.info/",
        "name": "World Air Quality Index Project"
      }
    ],
    "city": {
      "geo": [
        19.42461,
        -99.119594
      ],
      "name": "Merced, M\u00e9xico, Mexico",
      "url": "https://aqicn.org/city/mexico/mexico/merced",
      "location": ""
    },
    "dominentpol": "o3",
    "iaqi": {
      "co": {
        "v": 6.7
      },
      "dew": {
        "v": 14
      },
      "h": {
        "v": 56
      },
      "no2": {
        "v": 14.9
      },
      

### 2.- Consultar por Geolocalización (latitud, longitud)

In [17]:
# Coordenadas de San Luis Potosí
latitud = "22.1565"
longitud = "-100.9855"

url_geo = f"{BASE_URL}/feed/geo:{latitud};{longitud}/?token={API_TOKEN}"

response_geo = requests.get(url_geo)
data_geo = response_geo.json()

print(json.dumps(data_geo, indent=2))

{
  "status": "ok",
  "data": {
    "aqi": 17,
    "idx": 13291,
    "attributions": [
      {
        "url": "http://sinaica.inecc.gob.mx/",
        "name": "INECC - Instituto Nacional de Ecolog&iacute;a y Cambio Clim&aacute;tico",
        "logo": "Mexico-INECC.png"
      },
      {
        "url": "https://waqi.info/",
        "name": "World Air Quality Index Project"
      }
    ],
    "city": {
      "geo": [
        22.15125,
        -100.99611111111
      ],
      "name": "Industriales Potosinos Asociados, San Luis Potos\u00ed Estatal, San Luis Potos\u00ed, Mexico",
      "url": "https://aqicn.org/city/mexico/san-luis-potosi/san-luis-potosi-estatal/industriales-potosinos-asociados",
      "location": ""
    },
    "dominentpol": "o3",
    "iaqi": {
      "co": {
        "v": 2.1
      },
      "dew": {
        "v": 13
      },
      "h": {
        "v": 50
      },
      "no2": {
        "v": 3.8
      },
      "o3": {
        "v": 16.8
      },
      "p": {
        "v": 1024.3
   

### 3.- Usar la API con ubicación actual "here"

In [18]:
# El endpoint "here" detecta automáticamente tu ubicación
url_aqui = f"{BASE_URL}/feed/here/?token={API_TOKEN}"

response_aqui = requests.get(url_aqui)
data_aqui = response_aqui.json()

if data_aqui.get("status") == "ok":
    nombre_estacion = data_aqui["data"]["city"]["name"]
    aqi = data_aqui["data"]["aqi"]
    print(f"La calidad del aire en la estación más cercana ({nombre_estacion}) es: {aqi} AQI.")
else:
    print("No se pudo determinar la calidad del aire para tu ubicación.")
    print(data_aqui)

La calidad del aire en la estación más cercana (DIF, San Luis Potosí, Mexico) es: 30 AQI.


### 4.- Buscar estaciones por nombre 

Puedo buscar estaciones que contengan una palabra clave. Por ejemplo, busquemos estaciones que incluyan "Parque".

In [19]:
palabra_clave = "Parque"

url_busqueda = f"{BASE_URL}/search/?token={API_TOKEN}&keyword={palabra_clave}"

response_busqueda = requests.get(url_busqueda)
data_busqueda = response_busqueda.json()

if data_busqueda.get("status") == "ok":
    print(f"Se encontraron {len(data_busqueda['data'])} estaciones con la palabra '{palabra_clave}':")
    for estacion in data_busqueda['data']:
        print(f"- Estación: {estacion['station']['name']}, AQI: {estacion['aqi']}")
else:
    print("La búsqueda no arrojó resultados.")

Se encontraron 11 estaciones con la palabra 'Parque':
- Estación: Parque de La Piedra, Canarias, Spain, AQI: 34
- Estación: Parque Kanata, Cochabamba, Bolivia, AQI: -
- Estación: Parque Zapatón, Cantabria, Spain, AQI: 26
- Estación: Parque D.Pedro II, São Paulo, Brazil, AQI: 36
- Estación: Parque O'Higgins, Chile, AQI: 42
- Estación: Parque de San Juan-Telde, Canarias, Spain, AQI: 35
- Estación: Parque La Granja-Sta Cruz de TF, Canarias, Spain, AQI: -
- Estación: Parque Central da Taipa, Macau; Parque Central da Taipa, Macau (氹仔區 (氹仔中央公園站)), AQI: 25
- Estación: Zelaieta Parque, Amorebieta-Etxano, País Vasco, Spain, AQI: 42
- Estación: Metrorrey, Nuevo Leon, Mexico, AQI: 71
- Estación: Base Aérea-Acuaparque, Cali, Colombia, AQI: -


### 5.- Búsqueda y obtención de datos completos de una estación

In [20]:
# 1. BUSCAR LA ESTACIÓN POR NOMBRE
#------------------------------------
# Define el nombre de la estación que quieres buscar.
# Puedes probar con otras como "Merced", "Ajusco", "UAM", etc.
nombre_estacion_buscar = "Pedregal"

print(f"--- Paso 1: Buscando estaciones que coincidan con '{nombre_estacion_buscar}' ---")

url_busqueda = f"{BASE_URL}/search/?token={API_TOKEN}&keyword={nombre_estacion_buscar}"
response_busqueda = requests.get(url_busqueda)
data_busqueda = response_busqueda.json()

# Verificamos si la búsqueda fue exitosa y si hubo resultados
if data_busqueda.get("status") == "ok" and data_busqueda.get("data"):
    # Tomamos el primer resultado de la lista
    primera_estacion_encontrada = data_busqueda["data"][0]
    
    # Extraemos la URL de la estación, que usaremos para la siguiente consulta
    url_estacion_especifica = primera_estacion_encontrada["station"]["url"]
    
    print(f"Estación encontrada: {primera_estacion_encontrada['station']['name']}")
    print(f"URL para consulta: {url_estacion_especifica}")

    # 2. OBTENER TODOS LOS DATOS DE ESA ESTACIÓN
    #-------------------------------------------------
    print("\n--- Paso 2: Obteniendo todos los datos de la estación encontrada ---")
    
    # El 'feed' requiere el nombre de la estación como aparece en su URL.
    # Por ejemplo, si la URL es "aqicn.org/city/mexico/pedregal", necesitamos "mexico/pedregal".
    # Usamos la URL que ya obtuvimos en la búsqueda.
    url_completa = f"{BASE_URL}/feed/{url_estacion_especifica}/?token={API_TOKEN}"
    
    response_completa = requests.get(url_completa)
    datos_completos = response_completa.json()

    # Si la segunda consulta fue exitosa, extraemos y mostramos los datos
    if datos_completos.get("status") == "ok":
        # Extraemos las coordenadas
        coordenadas = datos_completos["data"]["city"]["geo"]
        latitud = coordenadas[0]
        longitud = coordenadas[1]
        
        print(f"\n **Coordenadas:**")
        print(f"   - Latitud: {latitud}")
        print(f"   - Longitud: {longitud}")
        
        # Imprimimos TODOS los datos de la estación de forma ordenada
        print("\n **Datos Completos de la Estación:**")
        print(json.dumps(datos_completos["data"], indent=4, ensure_ascii=False))
        
    else:
        print("Error: No se pudieron obtener los datos completos para la estación.")
        print(datos_completos)
        
else:
    print(f"No se encontraron estaciones con el nombre '{nombre_estacion_buscar}'.")


--- Paso 1: Buscando estaciones que coincidan con 'Pedregal' ---
Estación encontrada: Pedregal, México, Mexico
URL para consulta: mexico/mexico/pedregal

--- Paso 2: Obteniendo todos los datos de la estación encontrada ---

 **Coordenadas:**
   - Latitud: 19.325146
   - Longitud: -99.204136

 **Datos Completos de la Estación:**
{
    "aqi": 63,
    "idx": 407,
    "attributions": [
        {
            "url": "http://www.aire.df.gob.mx/",
            "name": "SINAICA - Sistema Nacional de Información de la Calidad del Aire en Mexico",
            "logo": "Mexico-CuidaMexico.png"
        },
        {
            "url": "http://sinaica.inecc.gob.mx/",
            "name": "INECC - Instituto Nacional de Ecolog&iacute;a y Cambio Clim&aacute;tico",
            "logo": "Mexico-INECC.png"
        },
        {
            "url": "https://waqi.info/",
            "name": "World Air Quality Index Project"
        }
    ],
    "city": {
        "geo": [
            19.325146,
            -99.2041

### Buscar por nombre de estación y me devulve las coordenadas

In [34]:
# --- PARÁMETRO DE BÚSQUEDA ---
# Cambia "UAM" por cualquier estación que quieras encontrar
nombre_estacion = "secundaria"
# -----------------------------

#BASE_URL = "https://api.waqi.info"
print(f" Buscando la estación '{nombre_estacion}'...")

# 1. Usamos el endpoint de búsqueda
url_busqueda = f"{BASE_URL}/search/?token={API_TOKEN}&keyword={nombre_estacion}"
response_busqueda = requests.get(url_busqueda)
data_busqueda = response_busqueda.json()

# 2. Verificamos si la búsqueda tuvo resultados
if data_busqueda.get("status") == "ok" and data_busqueda.get("data"):
    # Obtenemos el identificador único de la estación (la parte final de su URL)
    station_id = data_busqueda["data"][0]["station"]["url"]
    
    # 3. Solicitamos los datos completos de esa estación para obtener las coordenadas
    url_feed = f"{BASE_URL}/feed/{station_id}/?token={API_TOKEN}"
    response_feed = requests.get(url_feed)
    data_feed = response_feed.json()
    
    if data_feed.get("status") == "ok":
        # Extraemos las coordenadas del objeto de datos
        coordenadas = data_feed["data"]["city"]["geo"]
        latitud = coordenadas[0]
        longitud = coordenadas[1]
        
        print("\n ¡Coordenadas encontradas!")
        print(f"    Latitud:  {latitud}")
        print(f"    Longitud: {longitud}")
    else:
        print(" Error: Se encontró la estación pero no se pudieron obtener sus datos.")
else:
    print(f" Error: No se encontró ninguna estación que coincida con '{nombre_estacion}'.")


 Buscando la estación 'secundaria'...

 ¡Coordenadas encontradas!
    Latitud:  20.692413
    Longitud: -101.364861
